# Perkenalan

Final Project RMT-052 group 2

Anggota:
1. Ahmad Mustofa Zakariya
2. Fajar Rachman
3. Farhan Hamid Lubis
4. Moses Imanuel Salim

# Import Libraries

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
import warnings
warnings.filterwarnings('ignore')

# Data load and cleaning

In [2]:
# Read df

df_ori = pd.read_csv("ecommerce_customer_behavior_dataset_v2.csv")

df_ori.head(20)

,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_000001-1,CUST_00001,2023-05-29,40,Male,Ankara,Books,29.18,1,0.00,29.18,Digital Wallet,Mobile,14,9,True,13,4
1,ORD_000001-2,CUST_00001,2023-10-12,40,Male,Ankara,Home & Garden,644.40,1,138.05,506.35,Credit Card,Desktop,14,8,True,6,2
2,ORD_000001-3,CUST_00001,2023-12-05,40,Male,Ankara,Sports,332.82,5,0.00,1664.10,Credit Card,Mobile,15,10,True,9,4
3,ORD_000002-1,CUST_00002,2023-05-11,33,Male,Istanbul,Food,69.30,5,71.05,275.45,Digital Wallet,Desktop,16,13,True,4,4
4,ORD_000002-2,CUST_00002,2023-06-16,33,Male,Istanbul,Beauty,178.15,3,0.00,534.45,Credit Card,Mobile,14,7,True,6,4
5,ORD_000003-1,CUST_00003,2023-02-27,42,Male,Konya,Toys,198.28,2,0.00,396.56,Credit Card,Tablet,10,9,False,6,2
6,ORD_000003-2,CUST_00003,2024-01-03,42,Male,Konya,Home & Garden,526.85,5,0.00,2634.25,Digital Wallet,Desktop,11,8,True,6,5
7,ORD_000004-1,CUST_00004,2024-02-13,53,Male,Izmir,Fashion,96.20,5,97.78,383.22,Credit Card,Desktop,16,15,False,4,5
8,ORD_000005-1,CUST_00005,2023-03-16,32,Male,Ankara,Home & Garden,533.67,3,0.00,1601.01,Bank Transfer,Mobile,12,8,False,5,5
9,ORD_000005-2,CUST_00005,2023-06-12,32,Male,Ankara,Toys,73.06,4,0.00,292.24,Credit Card,Mobile,13,12,True,7,2


In [3]:
df = df_ori.copy()

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17049 entries, 0 to 17048
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order_ID                  17049 non-null  object 
 1   Customer_ID               17049 non-null  object 
 2   Date                      17049 non-null  object 
 3   Age                       17049 non-null  int64  
 4   Gender                    17049 non-null  object 
 5   City                      17049 non-null  object 
 6   Product_Category          17049 non-null  object 
 7   Unit_Price                17049 non-null  float64
 8   Quantity                  17049 non-null  int64  
 9   Discount_Amount           17049 non-null  float64
 10  Total_Amount              17049 non-null  float64
 11  Payment_Method            17049 non-null  object 
 12  Device_Type               17049 non-null  object 
 13  Session_Duration_Minutes  17049 non-null  int64  
 14  Pages_

In [5]:
df.columns = df.columns.str.lower()

In [6]:
df["date"] = pd.to_datetime(df['date'])

In [7]:
# Pembuatan kolom is_churn

# ambil last purchase per customer
last_purchase = df.groupby('customer_id')['date'].max().reset_index()

# ambil cutoff date (tanggal terakhir di dataset)
cutoff = df['date'].max()

# hitung selisih hari
last_purchase['days_since_last'] = (cutoff - last_purchase['date']).dt.days

# buat churn label (90 hari threshold nya)
last_purchase['is_churn'] = last_purchase['days_since_last'].apply(lambda x: 1 if x > 90 else 0)

# gabungkan kembali ke dataset utama
df = df.merge(last_purchase[['customer_id', 'is_churn']], on='customer_id', how='left')

print(df.head())

       order_id customer_id       date  age gender      city product_category  \
0  ORD_000001-1  CUST_00001 2023-05-29   40   Male    Ankara            Books   
1  ORD_000001-2  CUST_00001 2023-10-12   40   Male    Ankara    Home & Garden   
2  ORD_000001-3  CUST_00001 2023-12-05   40   Male    Ankara           Sports   
3  ORD_000002-1  CUST_00002 2023-05-11   33   Male  Istanbul             Food   
4  ORD_000002-2  CUST_00002 2023-06-16   33   Male  Istanbul           Beauty   

   unit_price  quantity  discount_amount  total_amount  payment_method  \
0       29.18         1             0.00         29.18  Digital Wallet   
1      644.40         1           138.05        506.35     Credit Card   
2      332.82         5             0.00       1664.10     Credit Card   
3       69.30         5            71.05        275.45  Digital Wallet   
4      178.15         3             0.00        534.45     Credit Card   

  device_type  session_duration_minutes  pages_viewed  is_returning_

In [8]:
# Is churn vs not churn

churn_dist = df['is_churn'].value_counts(normalize=True) * 100

print(f"Not Churn: {churn_dist[0]:.2f}%")
print(f"Churn: {churn_dist[1]:.2f}%")

Not Churn: 61.45%
Churn: 38.55%


# Feature Engineering

In [9]:
# split cat col num col

num_columns = df.select_dtypes(include=np.number).columns.tolist()
cat_columns = df.select_dtypes(include=['object']).columns.tolist()

In [10]:
# Handle null

print(df.isnull().sum())

order_id                    0
customer_id                 0
date                        0
age                         0
gender                      0
city                        0
product_category            0
unit_price                  0
quantity                    0
discount_amount             0
total_amount                0
payment_method              0
device_type                 0
session_duration_minutes    0
pages_viewed                0
is_returning_customer       0
delivery_time_days          0
customer_rating             0
is_churn                    0
dtype: int64


In [11]:
# Handle duplicate

print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [12]:
# Buat handle cardinality

for i in cat_columns:
    print(f'Jumlah unique value dari kolom {i} : {df[i].nunique()}') # menghitung data yg unik dari kolom kategori
    print(f'Unique value dari kolom {i} : {df[i].unique()}') # menampilkan data kategori yang unik
    print('')

Jumlah unique value dari kolom order_id : 17049
Unique value dari kolom order_id : ['ORD_000001-1' 'ORD_000001-2' 'ORD_000001-3' ... 'ORD_005000-2'
 'ORD_005000-3' 'ORD_005000-4']

Jumlah unique value dari kolom customer_id : 5000
Unique value dari kolom customer_id : ['CUST_00001' 'CUST_00002' 'CUST_00003' ... 'CUST_04998' 'CUST_04999'
 'CUST_05000']

Jumlah unique value dari kolom gender : 3
Unique value dari kolom gender : ['Male' 'Female' 'Other']

Jumlah unique value dari kolom city : 10
Unique value dari kolom city : ['Ankara' 'Istanbul' 'Konya' 'Izmir' 'Kayseri' 'Bursa' 'Gaziantep' 'Adana'
 'Eskisehir' 'Antalya']

Jumlah unique value dari kolom product_category : 8
Unique value dari kolom product_category : ['Books' 'Home & Garden' 'Sports' 'Food' 'Beauty' 'Toys' 'Fashion'
 'Electronics']

Jumlah unique value dari kolom payment_method : 5
Unique value dari kolom payment_method : ['Digital Wallet' 'Credit Card' 'Bank Transfer' 'Debit Card'
 'Cash on Delivery']

Jumlah unique valu

In [13]:
# Split x dan y

x = df.drop(['is_churn'] , axis=1)
y = df.is_churn

x_train, x_test, y_train, y_test = train_test_split(x, y, random_state = 0)
print('X Train Size: ', x_train.shape)
print('X Test Size: ', x_test.shape)
print('y Train Size: ', y_train.shape)
print('y Test Size: ', y_test.shape)

X Train Size:  (12786, 18)
X Test Size:  (4263, 18)
y Train Size:  (12786,)
y Test Size:  (4263,)


In [14]:
# Num col dan cat col

num_columns = x_train.select_dtypes(include=np.number).columns.tolist()
cat_columns = x_train.select_dtypes(include=['object']).columns.tolist()

In [15]:
# Pemisahan nul col dan cat col pada train dan test set

x_train_num = x_train[num_columns]
x_train_cat = x_train[cat_columns]

x_test_num = x_test[num_columns]
x_test_cat = x_test[cat_columns]

In [16]:
# Cek skewness

skewness_results = x_train_num.skew()

normal_columns = []
skewed_columns = []
extreme_skewed_columns = []

for col, skewness in skewness_results.items():
    if skewness < -1.0 or skewness > 1.0:
        extreme_skewed_columns.append(col)
    elif abs(skewness) <= 0.5:
        normal_columns.append(col)
    else:
        skewed_columns.append(col)

print(f"Normal: {normal_columns}\nSkewed: {skewed_columns}\nExtreme Skewed: {extreme_skewed_columns}")

Normal: ['age', 'quantity', 'session_duration_minutes', 'pages_viewed']
Skewed: ['customer_rating']
Extreme Skewed: ['unit_price', 'discount_amount', 'total_amount', 'delivery_time_days']


In [17]:
# Cek persen outlier

def calculate_outlier_percentages_skew(df, columns, distance):
    for variable in columns:
        IQR = df[variable].quantile(0.75) - df[variable].quantile(0.25)
        lower_boundary = df[variable].quantile(0.25) - (IQR * distance)
        upper_boundary = df[variable].quantile(0.75) + (IQR * distance)

        outliers = df[(df[variable] < lower_boundary) | (df[variable] > upper_boundary)]
        outlier_percentage = len(outliers) / len(df) * 100

        print('Percentage of outliers in {}: {:.2f}%'.format(variable, outlier_percentage))

# Calcuate outlier percentages before handling
calculate_outlier_percentages_skew(x_train_num, skewed_columns, 1.5)
calculate_outlier_percentages_skew(x_train_num, extreme_skewed_columns, 3)

Percentage of outliers in customer_rating: 0.00%
Percentage of outliers in unit_price: 5.09%
Percentage of outliers in discount_amount: 12.18%
Percentage of outliers in total_amount: 6.53%
Percentage of outliers in delivery_time_days: 0.31%


In [18]:
# cek balance

y_train.value_counts()

is_churn
0    7852
1    4934
Name: count, dtype: int64

# Pipeline

In [19]:
# Pipeline

outlier_columns = ['unit_price', 'discount_amount', 'total_amount']
num_columns = [
    'age',
    'quantity',
    'session_duration_minutes',
    'pages_viewed',
    'delivery_time_days',
]
def drop_id(X):
    return X.drop(columns=['ID'], errors='ignore')

# Num pipeline
num_pipeline = Pipeline([
    ('winsorizer', Winsorizer(
        capping_method='iqr',
        tail='both',
        fold=1.5,
        variables=outlier_columns
    )),
    ('scaler', StandardScaler())
])

# Ordinal cat col
ordinal_columns = ['customer_rating']
ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder())
])

# Nominal cat col
nominal_columns = [
    'gender',
    'city',
    'product_category',
    'payment_method',
    'device_type'
]
nominal_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

drop_id_transformer = FunctionTransformer(drop_id, validate=False)
preprocessing_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_columns + outlier_columns),
    ('ordinal', ordinal_pipeline, ordinal_columns),
    ('nominal', nominal_pipeline, nominal_columns)
])

In [20]:
# Pipeline model training untuk semua model

pipe_logreg = Pipeline([('drop_id', drop_id_transformer),('preprocessing', preprocessing_pipeline),('model_logreg', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))])


# Model Training

In [21]:
pipe_logreg.fit(x_train, y_train)

Pipeline(steps=[('drop_id',
                 FunctionTransformer(func=<function drop_id at 0x0000029E38643CA0>)),
                ('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('winsorizer',
                                                                   Winsorizer(capping_method='iqr',
                                                                              fold=1.5,
                                                                              tail='both',
                                                                              variables=['unit_price',
                                                                                         'discount_amount',
                                                                                         'total_amount'])),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'quantity',
                                                   'session_du...
                                                   'total_amount']),
                                                 ('ordinal',
                                                  Pipeline(steps=[('ordinal',
                                                                   OrdinalEncoder())]),
                                                  ['customer_rating']),
                                                 ('nominal',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['gender', 'city',
                                                   'product_category',
                                                   'payment_method',
                                                   'device_type'])])),
                ('model_logreg',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [22]:
# Prediction

y_pred_train = pipe_logreg.predict(x_train)
y_pred_test = pipe_logreg.predict(x_test)

In [23]:
print('F1 score - Train Set  : ', f1_score(y_train, y_pred_train))
print('F1 Score - Test Set   : ', f1_score(y_test, y_pred_test))

F1 score - Train Set  :  0.4557119942969168
F1 Score - Test Set   :  0.4258271077908218


In [24]:
print('Classification Report Logreg: \n' ,classification_report(y_test, y_pred_test))

Classification Report Logreg: 
               precision    recall  f1-score   support

           0       0.61      0.50      0.55      2624
           1       0.38      0.49      0.43      1639

    accuracy                           0.50      4263
   macro avg       0.49      0.49      0.49      4263
weighted avg       0.52      0.50      0.50      4263



# Model Saving

In [25]:
with open('logreg_model.pkl', 'wb') as model_logreg_file:
    pickle.dump(pipe_logreg, model_logreg_file)
